# M14 on Kaggle: one sensitivity variant, one seed

**Set two things in the first code cell:** `SEED` and `VARIANT`.

- `dpub_benign` (tier 1, seeds 42, 43 and 44): D_pub is the 5,000 benign rows only (D2's ablation).
- Tier 2, seed 42 only: `clients10`, `clients20`, `alpha0.1`, `alpha1.0`, `alt_order`, `reseed`, `with_iat`, `no_number_weight`, `no_profiling`, `official_split`. What each one changes: `VARIANTS` in `scripts/m14_variant.py`.

The steps come from `scripts/m14_variant.py` (the same list the local dev check runs): the variant's partition (M2; a variant that only changes features or the timeline rebuilds the headline partition and restores its figures), E1 (M3 + M4), E3 with local + rehearsal (M7), M9's K sweep and `e4_added`. Everything is written under `reports/mX/full/<VARIANT>/seed<SEED>/`; the headline reports are not touched. About 5 to 6 h of CPU per run (estimated from the `dpub_benign` runs plus M3 and M4, not measured).

It needs nothing from the other M14 notebooks, so several variants can run at the same time.

**Attach as input:** the private Kaggle dataset with the M1 `processed/` files.
**Settings:** Internet *On*. Accelerator: *None*.
**Run:** *Save Version → Save & Run All (Commit)*.

Each step runs in its own process and the zip is rewritten after each one. `exit code: 137` means the process was killed (usually out of memory).

**Output:** `/kaggle/working/m14_<VARIANT>_seed<SEED>_outputs.zip`. Download it and bring it back, also after a failure.


In [ ]:
SEED = 42                  # dpub_benign: 42, 43 or 44; tier 2: 42
VARIANT = "clients10"
import glob, os, shlex, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src"); sys.path.insert(0, f"{REPO}/scripts")
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists("scripts/m14_variant.py"), "This clone predates tier 2. Restart the session so it clones the latest main."
from m14_variant import VARIANTS, commands
assert VARIANT in VARIANTS, sorted(VARIANTS)
assert SEED in (42, 43, 44), SEED
assert glob.glob("/kaggle/input/**/data_manifest.json", recursive=True), "The processed dataset is not attached: Add Input -> your M1 processed dataset."
TAG = f"{VARIANT}/seed{SEED}"
CMDS = commands(VARIANT, "full", SEED, "kaggle")
print("\n".join(CMDS))


In [ ]:
import shutil
REPS = [f"reports/m2/full/{VARIANT}"] + [f"reports/{m}/full/{TAG}" for m in ("m3", "m4", "m7", "m9")]
OUT = f"/kaggle/working/m14_{VARIANT}_seed{SEED}_outputs"
def pack():
    stage = "/kaggle/working/m14_stage"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage)
    for rep in REPS:
        if os.path.isdir(rep):
            shutil.copytree(rep, f"{stage}/{rep}", ignore=shutil.ignore_patterns("checkpoints", "seed4*"), dirs_exist_ok=True)
    for r in sorted(glob.glob("/kaggle/working/results/runs/*")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}", ignore=shutil.ignore_patterns("models"))
    shutil.make_archive(OUT, "zip", stage)
    shutil.rmtree(stage)
    print("zip written:", OUT + ".zip")

def step(cmd):
    print("=====", cmd, flush=True)
    rc = subprocess.run(shlex.split(cmd)).returncode
    print("exit code:", rc, flush=True)
    pack()
    return rc


In [ ]:
for c in CMDS:
    step(c)


In [ ]:
# What finished, and whether every run has per-client counts
for m in ["m4", "m7", "m9"]:
    f = f"reports/{m}/full/{TAG}/confusions.csv"
    ok = os.path.exists(f) and "csv_client" in open(f).read()
    print(f"{m}: {'per-client counts present' if ok else 'MISSING (see the exit codes above)'}")
print(subprocess.run(["git", "status", "--short"], capture_output=True, text=True).stdout[:3000])
